In [1]:
# se importan librerias para la carga del archivo y manejo de cadenas de texto
import pandas as pd
import re
import numpy as np


# se importa librería necesaria para aplicar la fórmula Haversine para el cálculo
# de la distancia entre municipio y municipiosede
import math
import requests

# Cargar el dataset original
df = pd.read_csv('reporte_citas_medicas_original.csv', low_memory=False)

# **Eliminación del día festivo**

In [2]:
# Convertimos fechacita temporalmente a formato fecha
fecha = pd.to_datetime(df["fechacita"], format="%Y-%m-%d")

# Fecha excepcional que será excluida del análisis
fecha_excepcional = pd.Timestamp("2026-07-13")

# Validamos cuántos registros corresponden a esa fecha
registros_eliminar = df[fecha == fecha_excepcional]

print("Registros encontrados:", len(registros_eliminar))
print("\nEstado de las citas:")
print(registros_eliminar["estado_cita"].value_counts())

# Eliminamos los registros del 13 de julio de 2026
df = df[fecha != fecha_excepcional].copy()

# Reiniciamos el índice
df.reset_index(drop=True, inplace=True)

print("\nRegistros restantes:", len(df))

Registros encontrados: 254

Estado de las citas:
estado_cita
CITA CANCELADA    242
INCUMPLIDO         12
Name: count, dtype: int64

Registros restantes: 56411


# **variable programacronico**

In [3]:
# Reemplazamos los valores vacíos de programacronico por "Ninguno"
df["programacronico"] = df["programacronico"].fillna("Ninguno")

# Validamos el resultado
print(df["programacronico"].value_counts(dropna=False))

programacronico
Ninguno                51199
Hipertension            2877
Diabetes                1211
Nefroproteccion         1086
Sindrome Metabolico       38
Name: count, dtype: int64


# **variable convenio nombre**

In [4]:
total_inicial = len(df)

# Limpiar columna previa si ya existía para evitar cruces
if 'grupo_convenionombre' in df.columns:
    df = df.drop(columns=['grupo_convenionombre'])

# 2. Función de agrupación con la categoría Null preparada para el futuro
def agrupar_convenios(valor):
    # Si viene un nulo real de Pandas (NaN / None) o una celda vacía, va a Null
    if pd.isna(valor):
        return ''

    val_str = str(valor).strip()
    if val_str == '' or val_str.lower() == 'nan':
        return ''

    val = re.sub(r'\s+', ' ', val_str).upper()

    # Reglas de clasificación estándar
    if 'NUEVA EPS' in val and ('SUB' in val or 'SUBSIDIADO' in val):
        return 'Nueva EPS Subsidiado'
    elif 'NUEVA EPS' in val and 'CONT' in val:
        return 'Nueva EPS Contributivo'
    elif 'PROTEGER' in val and 'SUBSIDIADO' in val:
        return 'Proteger EPS Subsidiado'
    elif 'PROTEGER' in val and 'CONTRIBUTIVO' in val:
        return 'Proteger EPS Contributivo'
    elif 'COOSALUD' in val:
        return 'Coosalud'
    elif 'PARTICULAR' in val:
        return 'Particular'
    elif 'SANITAS' in val:
        return 'Otros'
    else:
        return 'Otros'

# Aplicar la recategorización
df['grupo_convenionombre'] = df['convenionombre'].apply(agrupar_convenios)

# ==========================================
# 3. CONTROLES DE CALIDAD Y AUDITORÍA AUTOMÁTICA
# ==========================================
print("=== INFORME DE VALIDACIÓN: CONVENIOS (CON NULL PREPARADO) ===")
assert len(df) == total_inicial, "¡Error! Se perdieron o duplicaron filas."
print("✔ Integridad de registros: OK (Sin pérdida de filas).")

# Auditoría automática de la categoría 'Particular'
originales_particular = df[df['grupo_convenionombre'] == 'Particular']['convenionombre'].unique()
print(f"\n🔍 Valores originales clasificados como 'Particular':")
print(originales_particular)

# Auditoría automática de la categoría 'Otra EPS'
originales_otra_eps = df[df['grupo_convenionombre'] == 'Otra EPS']['convenionombre'].unique()
print(f"\n🔍 Valores originales clasificados como 'Otra EPS':")
print(originales_otra_eps)

# Auditoría automática de la categoría 'Null' (Si no hay nulos en este archivo, mostrará un arreglo vacío)
originales_null = df[df['grupo_convenionombre'] == 'Null']['convenionombre'].unique()
print(f"\n🔍 Valores originales clasificados como 'Null':")
print(originales_null)

# Resumen ejecutivo de distribución por categoría
print("\n📊 Resumen de registros por categoría agrupada:")
print(df['grupo_convenionombre'].value_counts())


=== INFORME DE VALIDACIÓN: CONVENIOS (CON NULL PREPARADO) ===
✔ Integridad de registros: OK (Sin pérdida de filas).

🔍 Valores originales clasificados como 'Particular':
['PARTICULAR']

🔍 Valores originales clasificados como 'Otra EPS':
[]

🔍 Valores originales clasificados como 'Null':
[]

📊 Resumen de registros por categoría agrupada:
grupo_convenionombre
Nueva EPS Subsidiado         31587
Nueva EPS Contributivo       21928
Coosalud                      1548
Proteger EPS Subsidiado       1339
Proteger EPS Contributivo        7
Particular                       1
Otros                            1
Name: count, dtype: int64


# **variable municipio**

In [5]:
total_inicial = len(df)

# Si la columna ya existía de corridas previas en el archivo, la eliminamos para evitar conflictos
if 'grupo_municipio' in df.columns:
    df = df.drop(columns=['grupo_municipio'])

# Diccionario completo de coordenadas base (Latitud, Longitud)
COORDINADAS = {
    'ACHI': (8.5772, -74.5583),
    'AGUSTIN CODAZZI': (10.0028, -73.2322),
    'AMAGA': (6.0422, -75.7094),
    'APARTADO': (7.8858, -76.6358),
    'ARBOLETES': (8.8503, -76.4358),
    'AYAPEL': (8.3142, -75.1436),
    'BARRANCABERMEJA': (7.0653, -73.8547),
    'BARRANCO DE LOBA': (8.9556, -74.1083),
    'BARRANCO MINAS': (2.5833, -69.8333),
    'BARRANQUILLA': (10.9685, -74.7813),
    'BELLO': (6.3375, -75.5575),
    'BOGOTA, D.C.': (4.7110, -74.0721),
    'BUENAVISTA': (9.0711, -75.4597),
    'BURITICA': (6.7411, -75.9189),
    'CACERES': (7.5794, -75.4356),
    'CAIMITO': (8.9667, -75.6167),
    'CALI': (3.4516, -76.5320),
    'CAMPO DE LA CRUZ': (10.3756, -74.8875),
    'CARTAGENA': (10.3910, -75.4794),
    'CAUCASIA': (7.9856, -75.1953),
    'CERETE': (8.8878, -75.7831),
    'CHALAN': (9.5372, -75.3333),
    'CHARALA': (6.2975, -73.1611),
    'CHIMA': (8.7306, -75.5186),
    'CHINU': (9.1114, -74.8778),
    'COELLO': (4.2936, -74.9242),
    'COLOSO': (9.5189, -75.3619),
    'CORDOBA': (5.4167, -75.7833),
    'COROZAL': (9.3094, -75.2975),
    'COTA': (4.8078, -74.0953),
    'COVEÑAS': (9.4031, -75.6942),
    'CUCUTA': (7.8939, -72.5078),
    'EL BAGRE': (7.5936, -74.8072),
    'EL BANCO': (9.0019, -73.9786),
    'EL CARMEN DE BOLIVAR': (9.7158, -75.1239),
    'EL ROBLE': (9.2319, -75.3789),
    'ENVIGADO': (6.1694, -75.5861),
    'EXTRANJERO(Venezuela, Ecuador, otros)': (0.0, 0.0),
    'GACHANCIPA': (4.9819, -73.8867),
    'GALAPA': (10.8931, -74.8842),
    'GALERAS': (9.1583, -75.1431),
    'GUADALAJARA DE BUGA': (3.9008, -76.3044),
    'GUAMO': (4.0178, -74.9750),
    'GUARANDA': (8.4419, -74.5361),
    'HATO COROZAL': (6.1633, -71.7614),
    'IBAGUE': (4.4389, -75.2322),
    'LA APARTADA': (7.9044, -75.4056),
    'LA UNION': (8.3283, -75.3314),
    'LIBANO': (4.9197, -75.0608),
    'LORICA': (9.2372, -75.8147),
    'LOS PALMITOS': (9.3783, -75.3853),
    'LURUACO': (10.6125, -75.1389),
    'MADRID': (4.7347, -74.2625),
    'MAGANGUE': (9.2514, -74.7536),
    'MAJAGUAL': (8.3583, -74.4250),
    'MALAMBO': (10.8572, -74.7736),
    'MARIA LA BAJA': (9.9881, -75.3083),
    'MARINILLA': (6.1775, -75.3408),
    'MEDELLIN': (6.2442, -75.5812),
    'MOMIL': (9.2392, -75.6322),
    'MOMPOS': (9.2408, -74.4267),
    'MONTELIBANO': (7.9864, -75.4042),
    'MONTERIA': (8.7479, -75.8814),
    'MORROA': (9.3242, -75.4853),
    'MOSQUERA': (4.7089, -74.2333),
    'OVEJAS': (9.5306, -75.2219),
    'PALMAR DE VARELA': (10.7303, -74.7508),
    'PALMITO': (9.3900, -75.5350),
    'PLANETA RICA': (8.8875, -75.5819),
    'PLATO': (9.7906, -74.7817),
    'POLONUEVO': (10.7681, -74.8617),
    'POPAYAN': (2.4419, -76.6064),
    'PUEBLO NUEVO': (8.0658, -75.4417),
    'PUERTO BERRIO': (6.4925, -74.4047),
    'RIOHACHA': (11.5444, -72.9072),
    'RIONEGRO': (6.1553, -75.3736),
    'RIOSUCIO': (5.4244, -75.7061),
    'SAHAGUN': (8.9431, -75.4456),
    'SAMPUES': (9.1864, -75.3828),
    'SAN ANDRES': (12.5847, -81.7006),
    'SAN ANTONIO DE PALMITO': (9.3900, -75.5350),
    'SAN BENITO': (8.9328, -75.5892),
    'SAN BENITO ABAD': (8.9328, -75.5892),
    'SAN BERNARDO DEL VIENTO': (9.3589, -75.9467),
    'SAN CARLOS': (8.7908, -75.6053),
    'SAN JACINTO': (9.8317, -75.1239),
    'SAN JUAN DE BETULIA': (9.3000, -75.4333),
    'SAN LUIS DE SINCE': (9.2886, -75.2536),
    'SAN MARCOS': (8.6478, -75.1417),
    'SAN ONOFRE': (9.7369, -75.5222),
    'SAN PABLO': (7.4878, -73.9161),
    'SAN PEDRO': (9.3972, -75.3217),
    'SAN PELAYO': (8.9592, -75.8361),
    'SAN ZENON': (9.3514, -74.3014),
    'SANTA MARTA': (11.2408, -74.1990),
    'SANTA ROSA DE CABAL': (4.8675, -75.6214),
    'SANTA ROSA DEL SUR': (7.9864, -74.0536),
    'SANTANDER DE QUILICHAO': (3.0136, -76.4864),
    'SANTIAGO DE TOLU': (9.5167, -75.5833),
    'SINCELEJO': (9.3047, -75.3978),
    'SOLEDAD': (10.9189, -74.7647),
    'SUCRE': (8.8167, -74.7167),
    'TAMALAMEQUE': (8.8417, -73.8375),
    'TIERRALTA': (8.1722, -76.1361),
    'TOCANCIPA': (4.9578, -73.9144),
    'TOLEDO': (7.3086, -72.4839),
    'TOLU VIEJO': (9.5083, -75.4344),
    'TURBACO': (10.3333, -75.4167),
    'VALLEDUPAR': (10.4631, -73.2532),
    'VILLAVICENCIO': (4.1420, -73.6266)
}

def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)
    a = math.sin(dlat / 2)**2 + math.cos(math.radians(lat1)) * math.cos(math.radians(lat2)) * math.sin(dlon / 2)**2
    c = 2 * math.asin(math.sqrt(a))
    return R * c

# 2. Función híbrida con regla para extranjeros y Fallback informado
def obtener_horas_hibrido(mun, sede, lat1, lon1, lat2, lon2):
    # REGLA EXTRANJERO: Si el municipio es extranjero, se asigna directamente más de 6 horas
    if 'EXTRANJERO' in mun:
        return 99.0, "REGLA EXTRANJERO (>6h)"

    if lat1 == lat2 and lon1 == lon2:
        return 0.0, "OSRM (Directo)"

    url = f"http://router.project-osrm.org/route/v1/driving/{lon1},{lat1};{lon2},{lat2}?overview=false"

    try:
        response = requests.get(url, timeout=4)
        if response.status_code == 200:
            data = response.json()
            if data.get('code') == 'Ok':
                segundos = data['routes'][0]['duration']
                return segundos / 3600.0, "OSRM"
    except Exception:
        pass

    # Fallback si OSRM no responde
    dist_km = haversine(lat1, lon1, lat2, lon2)
    horas_fallback = dist_km / 50.0
    print(f"⚠️ [FALLBACK ACTIVADO] Sin conexión OSRM para '{mun} -> {sede}'. Usando cálculo geodésico.")
    return horas_fallback, "FALLBACK (Haversine)"

def categorizar_horas(horas):
    if horas > 24.0: # Identificador lógico para la regla de extranjero u otros casos extremos
        return 'mas de 6 horas'
    elif horas <= 1.0:
        return '0 a 1 horas'
    elif horas <= 2.0:
        return '1 a 2 horas'
    elif horas <= 4.0:
        return '2 a 4 horas'
    elif horas <= 6.0:
        return '4 a 6 horas'
    else:
        return 'mas de 6 horas'

print("=== PROCESANDO MATRIZ (RUTEO REAL + REGLA EXTRANJERO + FALLBACK) ===")

pares_unicos = df[['municipio', 'municipiosede']].drop_duplicates().copy()
print(f"Total de pares únicos a procesar: {len(pares_unicos)}\n")

resultados_pares = []
contador_osrm = 0
contador_fallback = 0
contador_extranjero = 0

for idx, row in pares_unicos.iterrows():
    mun = str(row['municipio']).strip().upper()
    sede = str(row['municipiosede']).strip().upper()

    if mun == sede:
        horas = 0.0
        metodo = "Idéntico"
    else:
        c_mun = COORDINADAS.get(mun)
        c_sede = COORDINADAS.get(sede)

        # Si es extranjero, las coordenadas pueden ser nulas o genéricas
        if 'EXTRANJERO' in mun:
            horas = 99.0
            metodo = "Extranjero"
            contador_extranjero += 1
        elif c_mun and c_sede and c_mun != (0.0, 0.0):
            horas, metodo = obtener_horas_hibrido(mun, sede, c_mun[0], c_mun[1], c_sede[0], c_sede[1])
            if "FALLBACK" in metodo:
                contador_fallback += 1
            elif "OSRM" in metodo:
                contador_osrm += 1
        else:
            horas = 3.0
            metodo = "Falta Coordenadas (Por defecto)"
            print(f"⚠️ [ADVERTENCIA] Municipio sin coordenadas registradas: '{mun}'. Asignando valor por defecto.")
            contador_fallback += 1

    cat = categorizar_horas(horas)
    resultados_pares.append({
        'municipio': row['municipio'],
        'municipiosede': row['municipiosede'],
        'grupo_municipio': cat
    })

print(f"\n--- RESUMEN DE PROCESAMIENTO DE PARES ---")
print(f"✔ Procesados vía OSRM (Ruta Real): {contador_osrm}")
print(f"🌍 Clasificados automáticamente como Extranjeros (>6h): {contador_extranjero}")
print(f"⚠️ Procesados mediante Fallback de respaldo: {contador_fallback}")

# Cruzar con el DataFrame principal
df_maestro_pares = pd.DataFrame(resultados_pares)
df = df.merge(df_maestro_pares, on=['municipio', 'municipiosede'], how='left')

# ==========================================
# CONTROLES DE CALIDAD AUTOMÁTICOS
# ==========================================
assert len(df) == total_inicial, "¡Error! Se perdieron filas durante el merge."
assert df['grupo_municipio'].isnull().sum() == 0, "¡Error! Hay valores nulos."

print("\n✔ Integridad de datos validada correctamente.")
print("\n📊 Distribución resultante:")
print(df['grupo_municipio'].value_counts())


=== PROCESANDO MATRIZ (RUTEO REAL + REGLA EXTRANJERO + FALLBACK) ===
Total de pares únicos a procesar: 256

⚠️ [FALLBACK ACTIVADO] Sin conexión OSRM para 'SAN ANDRES -> SINCELEJO'. Usando cálculo geodésico.
⚠️ [FALLBACK ACTIVADO] Sin conexión OSRM para 'SAN ANDRES -> COROZAL'. Usando cálculo geodésico.

--- RESUMEN DE PROCESAMIENTO DE PARES ---
✔ Procesados vía OSRM (Ruta Real): 239
🌍 Clasificados automáticamente como Extranjeros (>6h): 3
⚠️ Procesados mediante Fallback de respaldo: 2

✔ Integridad de datos validada correctamente.

📊 Distribución resultante:
grupo_municipio
0 a 1 horas       51430
1 a 2 horas        3478
4 a 6 horas         806
2 a 4 horas         494
mas de 6 horas      203
Name: count, dtype: int64


# **variable fechacita**

In [6]:
# Convertimos la fecha de la cita a formato fecha
fecha = pd.to_datetime(df["fechacita"], format="%Y-%m-%d")

# Crear día de la semana
DIAS = [
    "LUNES", "MARTES", "MIERCOLES",
    "JUEVES", "VIERNES", "SABADO", "DOMINGO"
]

df["dia_semana"] = fecha.dt.dayofweek.map(
    lambda i: DIAS[int(i)]
)

# Crear mes
MESES = [
    "ENERO", "FEBRERO", "MARZO", "ABRIL",
    "MAYO", "JUNIO", "JULIO", "AGOSTO",
    "SEPTIEMBRE", "OCTUBRE", "NOVIEMBRE", "DICIEMBRE"
]

df["mes"] = fecha.dt.month.map(
    lambda m: MESES[int(m) - 1]
)

# Validación
print("Día de la semana:")
print(df["dia_semana"].value_counts())

print("\nMes:")
print(df["mes"].value_counts())

Día de la semana:
dia_semana
MIERCOLES    13681
JUEVES       11763
MARTES       11620
VIERNES       8465
LUNES         8066
SABADO        2746
DOMINGO         70
Name: count, dtype: int64

Mes:
mes
SEPTIEMBRE    20351
JULIO         18995
AGOSTO        17065
Name: count, dtype: int64


# **variable horacita**

In [7]:
def franja(hora):
    if pd.isna(hora):
        return None
    if "06:00:00" <= hora <= "07:59:59": return "MADRUGADA"
    if "08:00:00" <= hora <= "11:59:59": return "MANIANA"
    if "12:00:00" <= hora <= "13:59:59": return "MEDIODIA"
    if "14:00:00" <= hora <= "17:59:59": return "TARDE"
    if "18:00:00" <= hora <= "23:30:00": return "NOCHE"
    return "FUERA_DE_HORARIO"

df["grupo_horacita"] = df["horacita"].apply(franja)
df["grupo_horacita"].value_counts(dropna=False)

,count
grupo_horacita,
MANIANA,21860
TARDE,15479
MADRUGADA,11524
MEDIODIA,6833
NOCHE,709
FUERA_DE_HORARIO,6


# **variable edad**

In [8]:
def grupo_edad(edad):
    if pd.isna(edad):
        return None
    if edad < 0 or edad > 150: return "EDAD_FUERA_DE_RANGO"
    if edad <= 5:  return "PRIMERA_INFANCIA"
    if edad <= 11: return "INFANCIA"
    if edad <= 17: return "ADOLESCENCIA"
    if edad <= 28: return "JUVENTUD"
    if edad <= 59: return "ADULTEZ"
    return "VEJEZ"

df["grupo_edad"] = pd.to_numeric(df["edadanios"]).apply(grupo_edad)
df["grupo_edad"].value_counts(dropna=False)

,count
grupo_edad,
VEJEZ,19817
ADULTEZ,18383
INFANCIA,5270
JUVENTUD,5077
PRIMERA_INFANCIA,4894
ADOLESCENCIA,2970


# **variable medicoespecialidad**

In [9]:
# Normalizamos temporalmente los nombres de las especialidades (quitar espacios dobles)
especialidad_norm = (
    df["medicoespecialidad"]
    .astype("string")
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
    .str.upper()
)

#definición de la regla de agrupación
mapa_especialidades = {
    "PSIQUIATRIA": "PSIQUIATRIA",
    "UROLOGIA": "UROLOGIA",
    "NUTRICION Y DIETETICA": "NUTRICION Y DIETETICA",

    "CIRUGIA GENERAL": "CIRUGIA Y ORTOPEDIA",
    "ORTOPEDIA Y TRAUMATOLOGIA": "CIRUGIA Y ORTOPEDIA",

    "OTORRINOLARINGOLOGIA": "OTORRINOLARINGOLOGIA",

    "CARDIOLOGIA": "MEDICINA SUBESPECIALIZADA",
    "ENDOCRINOLOGIA": "MEDICINA SUBESPECIALIZADA",
    "NEFROLOGIA": "MEDICINA SUBESPECIALIZADA",

    "NEUROLOGIA": "NEUROLOGIA",
    "AUXILIAR ENFERMERIA": "AUXILIAR ENFERMERIA",
    "MEDICINA INTERNA": "MEDICINA INTERNA",
    "GINECOLOGIA Y OBSTETRICIA": "GINECOLOGIA Y OBSTETRICIA",
    "PEDIATRIA": "PEDIATRIA",

    "ODONTOLOGIA GENERAL": "ODONTOLOGIA GENERAL Y PREVENTIVA",
    "HIGIENE ORAL": "ODONTOLOGIA GENERAL Y PREVENTIVA",

    "MEDICINA GENERAL": "MEDICINA GENERAL",

    "FISIOTERAPIA": "REHABILITACION Y TERAPIAS",
    "TERAPIA OCUPACIONAL": "REHABILITACION Y TERAPIAS",
    "FONOAUDIOLOGIA": "REHABILITACION Y TERAPIAS",

    "PSICOLOGIA": "PSICOLOGIA Y TRABAJO SOCIAL",
    "TRABAJO SOCIAL": "PSICOLOGIA Y TRABAJO SOCIAL",

    "ENFERMERIA": "ENFERMERIA",
    "DERMATOLOGIA": "DERMATOLOGIA",

    "ODONTOPEDIATRIA": "ODONTOLOGIA ESPECIALIZADA",
    "ENDODONCIA": "ODONTOLOGIA ESPECIALIZADA",
    "CIRUGIA ORAL": "ODONTOLOGIA ESPECIALIZADA",
    "REHABILITACION ORAL": "ODONTOLOGIA ESPECIALIZADA",

    "PATOLOGO": "PATOLOGIA"
}

#creamos la nueva columna
df["grupo_medicoespecialidad"] = especialidad_norm.map(mapa_especialidades)

# Verificar si quedó alguna especialidad sin clasificar
no_clasificadas = df[
    especialidad_norm.notna() &
    df["grupo_medicoespecialidad"].isna()
]["medicoespecialidad"].value_counts()

print(no_clasificadas)

#validamos la variabla agrupada
print(df["grupo_medicoespecialidad"].value_counts(dropna=False))

Series([], Name: count, dtype: int64)
grupo_medicoespecialidad
REHABILITACION Y TERAPIAS           13022
AUXILIAR ENFERMERIA                 11654
MEDICINA INTERNA                     8163
PSICOLOGIA Y TRABAJO SOCIAL          3939
GINECOLOGIA Y OBSTETRICIA            3723
PEDIATRIA                            3541
CIRUGIA Y ORTOPEDIA                  2080
NUTRICION Y DIETETICA                1899
MEDICINA GENERAL                     1614
DERMATOLOGIA                         1265
MEDICINA SUBESPECIALIZADA            1201
UROLOGIA                             1201
NEUROLOGIA                            636
OTORRINOLARINGOLOGIA                  616
ODONTOLOGIA GENERAL Y PREVENTIVA      410
ENFERMERIA                            393
PSIQUIATRIA                           367
ODONTOLOGIA ESPECIALIZADA             361
PATOLOGIA                             326
Name: count, dtype: int64


# **variable pym**

In [10]:
pym_norm = (
    df["pym"]
    .astype("string")
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
    .str.upper()
)

df["grupo_pym"] = pym_norm.apply(
    lambda x: "CITA MORBILIDAD"
    if x == "CITA MORBILIDAD"
    else "PROMOCION, PREVENCION Y PROGRAMAS ESPECIALES"
)

df["grupo_pym"] = pym_norm.apply(
    lambda x: (
        pd.NA
        if pd.isna(x)
        else "CITA MORBILIDAD"
        if x == "CITA MORBILIDAD"
        else "PROMOCION, PREVENCION Y PROGRAMAS ESPECIALES"
    )
)

#validamos
print(df["grupo_pym"].value_counts(dropna=False))

grupo_pym
CITA MORBILIDAD                                 47644
PROMOCION, PREVENCION Y PROGRAMAS ESPECIALES     8767
Name: count, dtype: int64


# **variable codigocups**

In [11]:
# Normalizamos la variable codigocups para evitar problemas
# por espacios o por diferencias en el tipo de dato.
cups_norm = (
    df["codigocups"]
    .astype("string")
    .str.strip()
)

# Diccionario de agrupación de los 54 códigos CUPS
mapa_cups = {

    # 1. LABORATORIO CLINICO
    "102": "LABORATORIO CLINICO",

    # 2. PATOLOGIA
    "100": "PATOLOGIA",

    # 3. PROCEDIMIENTOS DE ENFERMERIA
    "890205": "PROCEDIMIENTOS DE ENFERMERIA",
    "890305": "PROCEDIMIENTOS DE ENFERMERIA",

    # 4. TERAPIAS Y REHABILITACION
    "890210": "TERAPIAS Y REHABILITACION",
    "890211": "TERAPIAS Y REHABILITACION",
    "890213": "TERAPIAS Y REHABILITACION",
    "890310": "TERAPIAS Y REHABILITACION",
    "890311": "TERAPIAS Y REHABILITACION",
    "890313": "TERAPIAS Y REHABILITACION",

    # 5. ODONTOLOGIA GENERAL Y PREVENTIVA
    "101": "ODONTOLOGIA GENERAL Y PREVENTIVA",
    "890203": "ODONTOLOGIA GENERAL Y PREVENTIVA",
    "890303": "ODONTOLOGIA GENERAL Y PREVENTIVA",

    # 6. ODONTOLOGIA ESPECIALIZADA
    "890217": "ODONTOLOGIA ESPECIALIZADA",
    "890218": "ODONTOLOGIA ESPECIALIZADA",
    "890220": "ODONTOLOGIA ESPECIALIZADA",
    "890224": "ODONTOLOGIA ESPECIALIZADA",
    "890317": "ODONTOLOGIA ESPECIALIZADA",
    "890318": "ODONTOLOGIA ESPECIALIZADA",
    "890320": "ODONTOLOGIA ESPECIALIZADA",
    "890324": "ODONTOLOGIA ESPECIALIZADA",

    # 7. CONSULTAS CLINICAS Y OTROS PROFESIONALES
    "890201": "CONSULTAS CLINICAS Y OTROS PROFESIONALES",
    "890206": "CONSULTAS CLINICAS Y OTROS PROFESIONALES",
    "890208": "CONSULTAS CLINICAS Y OTROS PROFESIONALES",
    "890209": "CONSULTAS CLINICAS Y OTROS PROFESIONALES",
    "890228": "CONSULTAS CLINICAS Y OTROS PROFESIONALES",
    "890235": "CONSULTAS CLINICAS Y OTROS PROFESIONALES",
    "890242": "CONSULTAS CLINICAS Y OTROS PROFESIONALES",
    "890244": "CONSULTAS CLINICAS Y OTROS PROFESIONALES",
    "890250": "CONSULTAS CLINICAS Y OTROS PROFESIONALES",
    "890266": "CONSULTAS CLINICAS Y OTROS PROFESIONALES",
    "890268": "CONSULTAS CLINICAS Y OTROS PROFESIONALES",
    "890274": "CONSULTAS CLINICAS Y OTROS PROFESIONALES",
    "890280": "CONSULTAS CLINICAS Y OTROS PROFESIONALES",
    "890282": "CONSULTAS CLINICAS Y OTROS PROFESIONALES",
    "890283": "CONSULTAS CLINICAS Y OTROS PROFESIONALES",
    "890284": "CONSULTAS CLINICAS Y OTROS PROFESIONALES",
    "890294": "CONSULTAS CLINICAS Y OTROS PROFESIONALES",

    "890301": "CONSULTAS CLINICAS Y OTROS PROFESIONALES",
    "890306": "CONSULTAS CLINICAS Y OTROS PROFESIONALES",
    "890308": "CONSULTAS CLINICAS Y OTROS PROFESIONALES",
    "890328": "CONSULTAS CLINICAS Y OTROS PROFESIONALES",
    "890335": "CONSULTAS CLINICAS Y OTROS PROFESIONALES",
    "890342": "CONSULTAS CLINICAS Y OTROS PROFESIONALES",
    "890344": "CONSULTAS CLINICAS Y OTROS PROFESIONALES",
    "890350": "CONSULTAS CLINICAS Y OTROS PROFESIONALES",
    "890366": "CONSULTAS CLINICAS Y OTROS PROFESIONALES",
    "890368": "CONSULTAS CLINICAS Y OTROS PROFESIONALES",
    "890374": "CONSULTAS CLINICAS Y OTROS PROFESIONALES",
    "890380": "CONSULTAS CLINICAS Y OTROS PROFESIONALES",
    "890382": "CONSULTAS CLINICAS Y OTROS PROFESIONALES",
    "890383": "CONSULTAS CLINICAS Y OTROS PROFESIONALES",
    "890384": "CONSULTAS CLINICAS Y OTROS PROFESIONALES",
    "890394": "CONSULTAS CLINICAS Y OTROS PROFESIONALES"
}

# Creamos la nueva variable agrupada
df["grupo_codigocups"] = cups_norm.map(mapa_cups)

# Verificamos si quedó algún CUPS sin clasificar
no_clasificados = df[
    cups_norm.notna() &
    df["grupo_codigocups"].isna()
]["codigocups"].value_counts()

print("CUPS no clasificados:")
print(no_clasificados)

# Revisamos cuántos registros quedaron en cada grupo
print("\nDistribución de grupo_codigocups:")
print(df["grupo_codigocups"].value_counts(dropna=False))

CUPS no clasificados:
Series([], Name: count, dtype: int64)

Distribución de grupo_codigocups:
grupo_codigocups
CONSULTAS CLINICAS Y OTROS PROFESIONALES    30245
TERAPIAS Y REHABILITACION                   13022
LABORATORIO CLINICO                         11654
ODONTOLOGIA GENERAL Y PREVENTIVA              410
PROCEDIMIENTOS DE ENFERMERIA                  393
ODONTOLOGIA ESPECIALIZADA                     361
PATOLOGIA                                     326
Name: count, dtype: int64


In [12]:
# Revisamos la correspondencia entre el CUPS original
# y el grupo asignado
df[
    ["codigocups", "grupo_codigocups"]
].drop_duplicates().sort_values(
    by=["grupo_codigocups", "codigocups"]
)

,codigocups,grupo_codigocups
5363,890201,CONSULTAS CLINICAS Y OTROS PROFESIONALES
3061,890206,CONSULTAS CLINICAS Y OTROS PROFESIONALES
3448,890208,CONSULTAS CLINICAS Y OTROS PROFESIONALES
49298,890209,CONSULTAS CLINICAS Y OTROS PROFESIONALES
4157,890228,CONSULTAS CLINICAS Y OTROS PROFESIONALES
2759,890235,CONSULTAS CLINICAS Y OTROS PROFESIONALES
2159,890242,CONSULTAS CLINICAS Y OTROS PROFESIONALES
2043,890244,CONSULTAS CLINICAS Y OTROS PROFESIONALES
2375,890250,CONSULTAS CLINICAS Y OTROS PROFESIONALES
74,890266,CONSULTAS CLINICAS Y OTROS PROFESIONALES


# **verificamos df**

In [13]:
df.columns

Index(['agendacita_id', 'fechacita', 'horacita', 'fechaasignacion',
       'hora_inicio_consulta', 'fechaatencion', 'oportunidad',
       'id_paciente_anonimo', 'sexopaciente', 'edadanios', 'programacronico',
       'municipio', 'codigocups', 'id_medico_anonimo', 'medicoespecialidad',
       'diagnosticoprincipal', 'diagnosticorelacionado1',
       'diagnosticorelacionado2', 'diagnosticorelacionado3', 'causaexterna_id',
       'causaexternanombre', 'finalidad_id', 'finalidad', 'tipodiagnostico_id',
       'tipodiagnosticonombre', 'pym', 'modalidad', 'estado_consulta',
       'nota_no_asistencia', 'funcionario_nota_no_asistencia_anonimizado',
       'funcionarioasignacita_anonimo', 'convenionombre', 'funcionalidad',
       'rotulo', 'regimen', 'nombreips', 'nombresede', 'municipiosede',
       'turno_doble', 'identificador_agenda', 'nombre_estado', 'notacita',
       'fecha_ingreso_cronico', 'tipocita', 'remision', 'poblacion_vulnerable',
       'estado_cita', 'grupo_convenionombre', 'g

# **guardamos variables**

In [14]:
# Aseguras que 'estado_cita' esté siempre al final de forma automática
if 'estado_cita' in df.columns:
    # 1. Sacamos todas las columnas excepto 'estado_cita'
    cols = [col for col in df.columns if col != 'estado_cita']
    # 2. Volvemos a agregar 'estado_cita' al final de la lista
    cols.append('estado_cita')
    # 3. Reordenamos el DataFrame
    df = df[cols]


#guardamos la variable en el csv, lo voy a llamar vkate
df.to_csv(
    "reporte_citas_medicas_transformacionvariables.csv",
    index=False,
    encoding="utf-8",
    na_rep=""
)